In [ ]:
!pip install -U bitsandbytes transformers datasets peft accelerate trl unsloth python-docx

In [ ]:
import os, json, random, math, re
from typing import List, Dict, Tuple, Optional

import torch
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments, AddedToken
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer

# Recommended for A100 throughput
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- paths ---
INPUT_JSON  = "/content/input_full.json"     # raw (no tags)
OUTPUT_JSON = "/content/output_full.json"    # gold (with <ul>/<h>/<e> in text_body)
OUTPUT_DIR  = "/content/markup_finetune"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- model/runtime ---
MODEL_NAME   = "unsloth/gpt-oss-20b"
SEED         = 3407
BF16         = is_bfloat16_supported()
LOAD_IN_4BIT = True

random.seed(SEED)

# Markup tags as special tokens (helps the model treat each tag as a single unit)
MARKUP_STRINGS = ['<ul>','</ul>','<h>','</h>','<e>','</e>']
ADDED_MARKUP_TOKENS = [AddedToken(t, lstrip=False, rstrip=False, normalized=False) for t in MARKUP_STRINGS]

# Instructions for inline markup (not JSON spans)
SPAN_INSTRUCTIONS = (
    "You identify and mark spans in competitive debate cards by inserting tags into the body text.\n\n"
    "Write ONLY the marked text with the following tags inserted at the correct character positions:\n"
    "- Underline: <ul>…</ul>\n"
    "- Highlight: <h>…</h>  (must be fully inside <ul>…</ul>)\n"
    "- Emphasis:  <e>…</e>  (must be fully inside <h>…</h>)\n\n"
    "Rules:\n"
    "- Partial-word spans are allowed (UL/H/E). Keep H inside UL, and E inside H.\n"
    "- Prefer contiguous spans; avoid many single-character spans unless necessary.\n"
    "- Avoid leading/trailing punctuation at span edges unless it is essential for meaning.\n"
    "- Do NOT add or delete words—copy the text exactly and only insert tags.\n"
    "- Output only the marked body text. No extra commentary.\n"
)

PROMPT_TEMPLATE = (
    "{inst}"
    "Tag: {tag}\n"
    "Cite: {cite}\n"
    "Text:\n{text}\n\n"
    "Marked text:\n"
)

In [ ]:
# Load paired input/output JSONs
with open(INPUT_JSON, encoding="utf-8") as f_in, open(OUTPUT_JSON, encoding="utf-8") as f_out:
    raw_cards   = json.load(f_in)
    gold_cards  = json.load(f_out)

assert isinstance(raw_cards, list) and isinstance(gold_cards, list), "Both files must be lists."
assert len(raw_cards) == len(gold_cards), f"Length mismatch: {len(raw_cards)} vs {len(gold_cards)}"

pairs = list(zip(raw_cards, gold_cards))
random.shuffle(pairs)

# 90/10 split
split = int(0.9 * len(pairs))
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

def make_example(inp: Dict, out: Dict):
    # We do NOT normalize for training—copy text exactly so positions match your gold tags
    prompt = PROMPT_TEMPLATE.format(
        inst=SPAN_INSTRUCTIONS,
        tag=inp["tag"],
        cite=inp["cite"],
        text=inp["text_body"],
    )
    marked = out["text_body"]  # already contains <ul>/<h>/<e> in the right places
    return {"text": prompt + marked}

train_ds = Dataset.from_list([make_example(i,o) for (i,o) in train_pairs])
val_ds   = Dataset.from_list([make_example(i,o) for (i,o) in val_pairs])

print(f"Built dataset: train={len(train_ds)}, val={len(val_ds)}")

In [ ]:
anchor = "Marked text:\n"
th = sum(anchor in t for t in train_ds["text"])
vh = sum(anchor in t for t in val_ds["text"])
print(f"Anchor present in {th}/{len(train_ds)} train, {vh}/{len(val_ds)} val")

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)

# Add special tokens for tags BEFORE measuring lengths
tok.add_special_tokens({"additional_special_tokens": ADDED_MARKUP_TOKENS})

# Ensure a pad token exists
if tok.pad_token is None:
    tok.pad_token = tok.eos_token or "[PAD]"

def measure_lengths(texts, batch=256):
    lens = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], add_special_tokens=True, padding=False, truncation=False)
        lens.extend(len(x) for x in enc["input_ids"])
    return np.array(lens, dtype=np.int32)

train_lens = measure_lengths(train_ds["text"])
def pct(a, p):
    try:  return int(np.percentile(a, p, method="nearest"))
    except TypeError: return int(np.percentile(a, p, interpolation="nearest"))

md  = int(np.median(train_lens))
p95 = pct(train_lens, 95)
p99 = pct(train_lens, 99)
mx  = int(train_lens.max())

HEADROOM = 256
CEILING  = 8192
recommended = min(int(math.ceil((p99 + HEADROOM) / 256) * 256), CEILING)
MAX_SEQ_LEN = max(4096, recommended)

print("Token length stats (train):")
print(f"  median: {md}, p95: {p95}, p99: {p99}, max: {mx}")
print(f"Recommended MAX_SEQ_LEN: {MAX_SEQ_LEN} (floor=4096, cap={CEILING}, headroom={HEADROOM})")

In [ ]:
def keep_indices(ds):
    lens = measure_lengths(ds["text"])
    return [i for i,L in enumerate(lens) if L <= MAX_SEQ_LEN]

t_keep = keep_indices(train_ds); v_keep = keep_indices(val_ds)
dropped_t = len(train_ds) - len(t_keep)
dropped_v = len(val_ds)   - len(v_keep)

train_ds = train_ds.select(t_keep)
val_ds   = val_ds.select(v_keep)

print(f"Filtered to MAX_SEQ_LEN={MAX_SEQ_LEN}")
print(f"Train kept {len(train_ds)} / dropped {dropped_t}")
print(f"Val   kept {len(val_ds)} / dropped {dropped_v}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    dtype           = None,             # auto fp16/bf16
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
)

# Add our markup tokens as true special tokens and resize embeddings
tokenizer.add_special_tokens({"additional_special_tokens": ADDED_MARKUP_TOKENS})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "[PAD]"

model.resize_token_embeddings(len(tokenizer))
tokenizer.padding_side = "right"

# Optional: safer decoding
tokenizer.clean_up_tokenization_spaces = False

In [ ]:
# Cell 7.1 — Verify that tags are single tokens (not split) in all relevant contexts
import re
from random import randint

probe = "foo <ul>word</ul> bar <h>x</h> <e>y</e>"
enc = tokenizer(probe, add_special_tokens=False)
toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print("Probe tokens:", toks)
print("All six tags intact in probe?:", all(t in toks for t in MARKUP_STRINGS))

# Check only the tags that are actually present in the prompt snippet
prompt_snippet = PROMPT_TEMPLATE.format(inst="", tag="t", cite="c", text="hello") + "<ul>ok</ul>"
enc2 = tokenizer(prompt_snippet, add_special_tokens=True)
toks2 = tokenizer.convert_ids_to_tokens(enc2["input_ids"])
present_tags = set(re.findall(r'(</?ul>|</?h>|</?e>)', prompt_snippet))
ok_present = all(any(tok == t for tok in toks2) for t in present_tags)
print("Tags present in prompt are single tokens?:", ok_present)
print("Present tags:", present_tags)

# Sanity check on one full training example (prompt + marked text)
idx = randint(0, len(train_ds)-1)
sample_text = train_ds[idx]["text"]
enc3 = tokenizer(sample_text, add_special_tokens=True, truncation=True, max_length=MAX_SEQ_LEN)
toks3 = tokenizer.convert_ids_to_tokens(enc3["input_ids"])
present_tags3 = set(re.findall(r'(</?ul>|</?h>|</?e>)', sample_text))
ok_present3 = all(any(tok == t for tok in toks3) for t in present_tags3)
print(f"Train sample {idx}: tags present are single tokens?:", ok_present3, "| num tags:", len(present_tags3))


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
from dataclasses import dataclass
from typing import Any, List, Dict, Optional

# custom collator to mask everything before "Marked text:\n"
RESPONSE_TEMPLATE = "Marked text:\n"
TEMPLATE_IDS = tokenizer(RESPONSE_TEMPLATE, add_special_tokens=False)["input_ids"]

def _find_sublist(hay: List[int], needle: List[int]) -> Optional[int]:
    n = len(needle)
    if n == 0: return 0
    for i in range(len(hay) - n + 1):
        if hay[i:i+n] == needle:
            return i
    return None

@dataclass
class CompletionOnlyCollator:
    tokenizer: Any
    max_seq_length: int
    template_ids: List[int]

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id

        if "input_ids" in features[0]:
            batch_ids = [f["input_ids"] for f in features]
            max_len = min(max(len(x) for x in batch_ids), self.max_seq_length)
            padded, attn = [], []
            for seq in batch_ids:
                seq = seq[:max_len]
                pad_len = max_len - len(seq)
                padded.append(seq + [pad_id] * pad_len)
                attn.append([1] * len(seq) + [0] * pad_len)
            input_ids = torch.tensor(padded, dtype=torch.long)
            attention_mask = torch.tensor(attn, dtype=torch.long)
        else:
            texts = [f["text"] for f in features]
            enc = self.tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.max_seq_length,
                add_special_tokens=True,
            )
            input_ids = enc["input_ids"]
            attention_mask = enc["attention_mask"]

        labels = input_ids.clone()
        for i in range(input_ids.size(0)):
            ids = input_ids[i].tolist()
            pos = _find_sublist(ids, self.template_ids)
            if pos is None:
                labels[i, :] = -100
            else:
                cutoff = pos + len(self.template_ids)
                labels[i, :cutoff] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,     # set 1 if OOM
    gradient_accumulation_steps=8,     # effective batch ~16
    num_train_epochs=1,                # one epoch per your choice
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    optim="adamw_8bit",                # if it errors, switch to "adamw_torch"
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    fp16=not BF16,
    bf16=BF16,
    report_to="none",
    do_eval=False,
)

collator = CompletionOnlyCollator(
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
    template_ids=TEMPLATE_IDS,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=None,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    data_collator=collator,
)

In [ ]:
train_result = trainer.train()
trainer.save_state()
print(train_result)

In [ ]:
save_dir = os.path.join(OUTPUT_DIR, "adapter")
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)
print(f"Saved LoRA adapters & tokenizer to: {save_dir}")

In [ ]:
# Utilities to parse generated tags -> spans, clean, then rebuild

def strip_tags_and_spans(tagged: str):
    """Return (raw_text, spans) by reading <ul>/<h>/<e> tags from a tagged string."""
    # Build raw text while tracking positions
    tag_re = re.compile(r'(</?ul>|</?h>|</?e>)')
    parts = tag_re.split(tagged)
    raw_chars = []
    pos = 0
    # stacks to track open tags: {type: [start_pos,...]}
    stacks = {"ul": [], "h": [], "e": []}
    spans = []
    for part in parts:
        if not part:
            continue
        if tag_re.fullmatch(part):
            is_close = part.startswith("</")
            typ = part.strip("</>")
            if not is_close:
                stacks[typ].append(pos)
            else:
                if stacks[typ]:
                    st = stacks[typ].pop()
                    if pos > st:
                        spans.append({"start": st, "end": pos, "type": typ})
            continue
        # normal text
        raw_chars.append(part)
        pos += len(part)
    raw_text = "".join(raw_chars)
    # any unclosed tags are ignored
    spans.sort(key=lambda s: (s["start"], -s["end"], {"ul":0,"h":1,"e":2}[s["type"]]))
    return raw_text, spans

# Validators/smoothers reused (trim punctuation, drop micro-spans, merge adjacent, enforce nesting)
# --- helpers from earlier ---
def clamp_span(s: int, e: int, L: int) -> Tuple[int,int]:
    s = max(0, min(s, L)); e = max(0, min(e, L))
    return (s, e) if e > s else (0, 0)

def merge_overlaps(ranges: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    if not ranges: return []
    ranges = sorted(ranges)
    merged = [list(ranges[0])]
    for s,e in ranges[1:]:
        ms, me = merged[-1]
        if s <= me: merged[-1][1] = max(me, e)
        else: merged.append([s,e])
    return [(s,e) for s,e in merged if e>s]

def union_of(ranges: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    return merge_overlaps(ranges)

def intersect(a: Tuple[int,int], b: Tuple[int,int]) -> Tuple[int,int]:
    s = max(a[0], b[0]); e = min(a[1], b[1])
    return (s,e) if e>s else (0,0)

def intersect_with_union(seg: Tuple[int,int], uni: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    out = []
    for u in uni:
        x = intersect(seg, u)
        if x != (0,0): out.append(x)
    return out

# smoothing knobs
SMOOTH = True
TRIM_PUNCT_EDGES = True
MIN_LEN = {"ul": 3, "h": 2, "e": 2}
DROP_PUNCT_ONLY = True
MERGE_ADJACENT = True

_WORD_INTERNAL = set(["-", "’", "'"])
_EDGE_TRIM_CHARS = set(list(" \t\n\r") + list(",.;:!?") + list("()[]{}") + ['"', "“", "”", "‘", "’", "'"])

def _is_word_char(ch: str) -> bool:
    return ch.isalnum() or ch in _WORD_INTERNAL

def _trim_edges(text: str, s: int, e: int) -> Tuple[int,int]:
    while s < e:
        ch = text[s]
        if ch in _EDGE_TRIM_CHARS and not _is_word_char(ch): s += 1
        else:
            if ch in _EDGE_TRIM_CHARS and not ch.isalnum(): s += 1
            else: break
    while e > s:
        ch = text[e-1]
        if ch in _EDGE_TRIM_CHARS and not _is_word_char(ch): e -= 1
        else:
            if ch in _EDGE_TRIM_CHARS and not ch.isalnum(): e -= 1
            else: break
    return (s,e) if e>s else (0,0)

def _merge_adjacent_only(ranges: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    if not ranges: return []
    rs = sorted(ranges)
    merged = [list(rs[0])]
    for s,e in rs[1:]:
        if s == merged[-1][1]: merged[-1][1] = e
        else: merged.append([s,e])
    return [(a,b) for a,b in merged if b>a]

def validate_and_fix_spans(text: str, spans: List[Dict]):
    L = len(text)
    typed = []
    for s in spans:
        if not isinstance(s, dict): continue
        t = s.get("type")
        if t not in ("ul","h","e"): continue
        try:
            st = int(s.get("start", -1)); ed = int(s.get("end", -1))
        except: continue
        if ed <= st: continue
        typed.append((t, st, ed))

    ul = [(st, ed) for t,st,ed in typed if t=="ul"]
    h  = [(st, ed) for t,st,ed in typed if t=="h"]
    e  = [(st, ed) for t,st,ed in typed if t=="e"]

    ul = [clamp_span(s,e,L) for s,e in ul if e>s]
    h  = [clamp_span(s,e,L) for s,e in h  if e>s]
    e  = [clamp_span(s,e,L) for s,e in e  if e>s]

    ul = merge_overlaps(ul)
    h  = merge_overlaps(h)
    e  = merge_overlaps(e)

    H_union = union_of(h)
    new_e = []
    for seg in e: new_e.extend(intersect_with_union(seg, H_union))
    e = merge_overlaps(new_e)

    UL_union = union_of(ul)
    new_h = []
    for seg in h: new_h.extend(intersect_with_union(seg, UL_union))
    h = merge_overlaps(new_h)

    if SMOOTH:
        def _keep(t,s,e):
            if (e-s) < MIN_LEN[t]: return False
            if DROP_PUNCT_ONLY:
                seg = text[s:e]
                return any(ch.isalnum() for ch in seg)
            return True
        ul = [(s,e) for (s,e) in ul if _keep("ul",s,e)]
        h  = [(s,e) for (s,e) in h  if _keep("h", s,e)]
        e  = [(s,e) for (s,e) in e  if _keep("e", s,e)]

        if TRIM_PUNCT_EDGES:
            ul = [_trim_edges(text,s,e) for (s,e) in ul]; ul = [(s,e) for (s,e) in ul if e>s]
            h  = [_trim_edges(text,s,e) for (s,e) in h];  h  = [(s,e) for (s,e) in h  if e>s]
            e  = [_trim_edges(text,s,e) for (s,e) in e];  e  = [(s,e) for (s,e) in e  if e>s]

        if MERGE_ADJACENT:
            ul = _merge_adjacent_only(ul)
            h  = _merge_adjacent_only(h)
            e  = _merge_adjacent_only(e)

        # Re-enforce nesting
        H_union = union_of(h)
        e2 = []
        for seg in e: e2.extend(intersect_with_union(seg, H_union))
        e = merge_overlaps(e2)
        UL_union = union_of(ul)
        h2 = []
        for seg in h: h2.extend(intersect_with_union(seg, UL_union))
        h = merge_overlaps(h2)

    out = []
    for s,e in ul: out.append({"start": s, "end": e, "type": "ul"})
    for s,e in h:  out.append({"start": s, "end": e, "type": "h"})
    for s,e in e:  out.append({"start": s, "end": e, "type": "e"})
    out.sort(key=lambda x: (x["start"], -x["end"], {"ul":0,"h":1,"e":2}[x["type"]]))
    return out

def rebuild_tagged_text(text: str, spans: List[Dict]):
    events = []
    for s in spans:
        t = s["type"]; a=s["start"]; b=s["end"]
        events.append((a, 0, f"<{t}>"))
        events.append((b, 1, f"</{t}>"))
    order = {"ul":0,"h":1,"e":2}
    def key(ev):
        pos, kind, tag = ev
        t = tag.strip("</>")
        return (pos, kind, order[t] if kind==0 else 2-order[t])
    events.sort(key=key)
    out, last = [], 0
    for pos, _, tag in events:
        out.append(text[last:pos]); out.append(tag); last = pos
    out.append(text[last:])
    return "".join(out)

@torch.inference_mode()
def predict_marked_text(card: Dict, max_new_tokens: int = 2048):
    prompt = PROMPT_TEMPLATE.format(
        inst=SPAN_INSTRUCTIONS,
        tag=card["tag"],
        cite=card["cite"],
        text=card["text_body"],
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    prompt_len = enc["input_ids"].shape[1]

    # Try generation; if empty, retry with larger budget
    out_ids = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = tokenizer.decode(out_ids[0, prompt_len:], skip_special_tokens=True).strip()

    # Parse -> spans -> clean -> rebuild
    raw_text, spans = strip_tags_and_spans(gen)
    spans = validate_and_fix_spans(raw_text, spans)
    cleaned = rebuild_tagged_text(raw_text, spans)
    return cleaned

In [ ]:
from docx import Document
from docx.enum.text import WD_COLOR_INDEX

def add_formatted_paragraph(doc, tagged_text: str):
    p = doc.add_paragraph()
    parts = re.split(r'(<ul>|</ul>|<h>|</h>|<e>|</e>)', tagged_text)
    in_ul = in_h = in_e = False
    for part in parts:
        if part == '<ul>': in_ul = True
        elif part == '</ul>': in_ul = False
        elif part == '<h>': in_h = True
        elif part == '</h>': in_h = False
        elif part == '<e>': in_e = True
        elif part == '</e>': in_e = False
        elif part:
            run = p.add_run(part)
            if in_ul: run.underline = True
            if in_h:  run.font.highlight_color = WD_COLOR_INDEX.BRIGHT_GREEN
            if in_e:  run.bold = True

def export_card_to_docx(tag: str, cite: str, marked_text: str, out_path: str):
    doc = Document()
    doc.add_heading(tag, level=4)
    doc.add_paragraph(cite)
    add_formatted_paragraph(doc, marked_text)
    doc.save(out_path)
    print(f"Wrote {out_path}")

# --- quick sample ---
sample_card = train_pairs[0][0] if train_pairs else raw_cards[0]
pred_marked = predict_marked_text(sample_card)
print(pred_marked[:800], "...\n")  # preview
# export_card_to_docx(sample_card["tag"], sample_card["cite"], pred_marked, "/content/sample_marked.docx")